In [1]:
# ============================================================
# CUSTOMER CHURN INTELLIGENCE
# 01 - DATA CLEANING & QUALITY ANALYSIS
# ============================================================

# %% [markdown]
# # Customer Churn Intelligence
# ## 01. Data Cleaning & Quality Analysis
#
# Dataset: IBM Telco Customer Churn
#
# Objective:
# - Load the raw customer dataset
# - Understand the structure and data types
# - Identify missing and duplicate records
# - Investigate data-quality issues
# - Clean and transform the dataset
# - Save a processed dataset for EDA and ML

# %% [markdown]
# ## 1. Import Libraries

# %%
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

# %% [markdown]
# ## 2. Load Raw Dataset

# %%
file_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

# %% [markdown]
# ## 3. Preview the Dataset

# %%
display(df.head())

# %% [markdown]
# ## 4. Dataset Structure

# %%
print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

# %% [markdown]
# ## 5. Data Types and Memory Information

# %%
df.info()

# %% [markdown]
# ## 6. Statistical Summary

# %%
display(df.describe(include="all").T)

# %% [markdown]
# ## 7. Check Missing Values

# %%
missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": (missing_values / len(df) * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values("Missing Values", ascending=False)

display(missing_summary)

# %% [markdown]
# ## 8. Check Empty Strings

# %%
empty_strings = (df == "").sum()

empty_summary = pd.DataFrame({
    "Empty Strings": empty_strings
})

empty_summary = empty_summary[
    empty_summary["Empty Strings"] > 0
]

display(empty_summary)

# %% [markdown]
# ## 9. Check Duplicate Customer IDs

# %%
duplicate_count = df["customerID"].duplicated().sum()

print(f"Duplicate customer IDs: {duplicate_count}")

# %% [markdown]
# ## 10. Check Duplicate Rows

# %%
duplicate_rows = df.duplicated().sum()

print(f"Duplicate complete rows: {duplicate_rows}")

# %% [markdown]
# ## 11. Churn Distribution

# %%
churn_counts = df["Churn"].value_counts()

print("Churn Counts:")
display(churn_counts)

# %%
churn_percentage = (
    df["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Churn Percentage:")
display(churn_percentage)

# %% [markdown]
# ## 12. Create a Working Copy
#
# We keep the original raw DataFrame unchanged.

# %%
clean_df = df.copy()

print("Working copy created.")

# %% [markdown]
# ## 13. Investigate TotalCharges
#
# TotalCharges should be numerical, but the raw dataset stores it as text.

# %%
print("Original TotalCharges dtype:")
print(clean_df["TotalCharges"].dtype)

# %%
print("Sample TotalCharges values:")
display(clean_df["TotalCharges"].head(20))

# %% [markdown]
# ## 14. Convert TotalCharges to Numeric
#
# Invalid/blank values are converted to NaN.

# %%
clean_df["TotalCharges"] = pd.to_numeric(
    clean_df["TotalCharges"],
    errors="coerce"
)

print("New TotalCharges dtype:")
print(clean_df["TotalCharges"].dtype)

# %% [markdown]
# ## 15. Identify Invalid TotalCharges Records

# %%
invalid_total_charges = clean_df[
    clean_df["TotalCharges"].isnull()
]

print(
    f"Records with invalid/missing TotalCharges: "
    f"{len(invalid_total_charges)}"
)

display(
    invalid_total_charges[
        [
            "customerID",
            "tenure",
            "MonthlyCharges",
            "TotalCharges",
            "Churn"
        ]
    ]
)

# %% [markdown]
# ## 16. Investigate Tenure of Missing TotalCharges

# %%
print("Tenure distribution for missing TotalCharges:")

display(
    clean_df.loc[
        clean_df["TotalCharges"].isnull(),
        "tenure"
    ].value_counts()
)

# %% [markdown]
# ## 17. Handle Missing TotalCharges
#
# For customers with tenure = 0, TotalCharges can logically be
# represented as 0 because they have not accumulated a billing period.
#
# We verify the records before applying the transformation.

# %%
missing_total = clean_df["TotalCharges"].isnull()

print("Missing TotalCharges records:")
print(clean_df.loc[missing_total, ["tenure", "MonthlyCharges"]])

# %%
zero_tenure_missing = (
    clean_df["TotalCharges"].isnull()
    & (clean_df["tenure"] == 0)
)

print(
    "Missing TotalCharges with zero tenure:",
    zero_tenure_missing.sum()
)

# %% [markdown]
# ## 18. Fill Missing TotalCharges

# %%
clean_df.loc[zero_tenure_missing, "TotalCharges"] = 0

# %%
remaining_missing_total = clean_df["TotalCharges"].isnull().sum()

print(
    "Remaining missing TotalCharges:",
    remaining_missing_total
)

# %% [markdown]
# ## 19. Verify Missing Values After Cleaning

# %%
missing_after_cleaning = clean_df.isnull().sum()

missing_after_summary = pd.DataFrame({
    "Missing Values": missing_after_cleaning,
    "Missing Percentage": (
        missing_after_cleaning / len(clean_df) * 100
    ).round(2)
})

missing_after_summary = missing_after_summary[
    missing_after_summary["Missing Values"] > 0
]

if len(missing_after_summary) == 0:
    print("No missing values remain.")
else:
    display(missing_after_summary)

# %% [markdown]
# ## 20. Convert Binary Columns
#
# Convert Yes/No variables into 1/0 for ML preparation later.
#
# We do this only for selected columns.

# %%
binary_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]

for column in binary_columns:
    clean_df[column] = clean_df[column].map({
        "Yes": 1,
        "No": 0
    })

# %%
display(clean_df[binary_columns].head())

# %% [markdown]
# ## 21. Check Unique Values in Categorical Columns

# %%
categorical_columns = clean_df.select_dtypes(
    include=["object"]
).columns

for column in categorical_columns:
    print(f"\n{column}")
    print(clean_df[column].value_counts(dropna=False))

# %% [markdown]
# ## 22. Validate Numerical Columns

# %%
numerical_columns = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

display(clean_df[numerical_columns].describe().T)

# %% [markdown]
# ## 23. Check for Negative Values

# %%
for column in numerical_columns:
    negative_count = (clean_df[column] < 0).sum()
    print(f"{column}: {negative_count} negative values")

# %% [markdown]
# ## 24. Check Final Duplicate Records

# %%
print(
    "Duplicate customer IDs:",
    clean_df["customerID"].duplicated().sum()
)

print(
    "Duplicate complete rows:",
    clean_df.duplicated().sum()
)

# %% [markdown]
# ## 25. Final Dataset Shape

# %%
print("Original shape:", df.shape)
print("Cleaned shape :", clean_df.shape)

# %% [markdown]
# ## 26. Final Preview

# %%
display(clean_df.head())

# %% [markdown]
# ## 27. Final Data Quality Report

# %%
quality_report = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Original Columns",
        "Cleaned Rows",
        "Cleaned Columns",
        "Duplicate Customer IDs",
        "Duplicate Rows",
        "Missing Values",
        "Churned Customers",
        "Non-Churned Customers"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        clean_df.shape[0],
        clean_df.shape[1],
        clean_df["customerID"].duplicated().sum(),
        clean_df.duplicated().sum(),
        clean_df.isnull().sum().sum(),
        clean_df["Churn"].sum(),
        (clean_df["Churn"] == 0).sum()
    ]
})

display(quality_report)

# %% [markdown]
# ## 28. Save Processed Dataset

# %%
output_path = "../data/processed/telco_customer_churn_clean.csv"

clean_df.to_csv(
    output_path,
    index=False
)

print(f"Cleaned dataset saved to: {output_path}")

# %% [markdown]
# ## 29. Reload the Processed Dataset and Verify

# %%
verification_df = pd.read_csv(output_path)

print("Processed dataset successfully reloaded.")
print("Shape:", verification_df.shape)

display(verification_df.head())

# %% [markdown]
# # Data Cleaning Completed
#
# The processed dataset is now ready for:
#
# 1. Exploratory Data Analysis
# 2. SQL analysis
# 3. Feature engineering
# 4. Machine learning
# 5. Customer churn prediction
# 6. Dashboard development

Libraries imported successfully.
Dataset loaded successfully.
Rows    : 7,043
Columns : 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Dataset Shape:
(7043, 21)

Column Names:
01. customerID
02. gender
03. SeniorCitizen
04. Partner
05. Dependents
06. tenure
07. PhoneService
08. MultipleLines
09. InternetService
10. OnlineSecurity
11. OnlineBackup
12. DeviceProtection
13. TechSupport
14. StreamingTV
15. StreamingMovies
16. Contract
17. PaperlessBilling
18. PaymentMethod
19. MonthlyCharges
20. TotalCharges
21. Churn
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Missing Values,Missing Percentage


...

SeniorCitizen: 0 negative values
tenure: 0 negative values
MonthlyCharges: 0 negative values
TotalCharges: 0 negative values
Duplicate customer IDs: 0
Duplicate complete rows: 0
Original shape: (7043, 21)
Cleaned shape : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


,Metric,Value
0,Original Rows,7043
1,Original Columns,21
2,Cleaned Rows,7043
3,Cleaned Columns,21
4,Duplicate Customer IDs,0
5,Duplicate Rows,0
6,Missing Values,0
7,Churned Customers,1869
8,Non-Churned Customers,5174


Cleaned dataset saved to: ../data/processed/telco_customer_churn_clean.csv
Processed dataset successfully reloaded.
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1
